# XXZ Chain Bayesian Analysis

**Paper:** "Robust Bayesian Inference Protocol for Finite-Size Scaling"

**Author:** Shi Yan (施延)

This notebook reproduces the main results from the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
sys.path.append('../src')
from scaling_analysis import BayesianScalingAnalyzer, load_data_from_csv

# Set plot style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 1. Load DMRG Data

In [ ]:
# Load data from CSV
N, S_A, sigma_num, sigma_trunc = load_data_from_csv('../data/XXZ_OBC_data.csv')
sigma_total = np.sqrt(sigma_num**2 + sigma_trunc**2)

print(f"Data points: n = {len(N)}")
print(f"System sizes: N = {N[0]} to {N[-1]}")
print(f"\nFirst 5 data points:")
for i in range(5):
    print(f"N={N[i]:3d}: S_A={S_A[i]:.4f} ± {sigma_total[i]:.4f}")

## 2. Visualize Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: S_A vs ln(N)
ax = axes[0]
ax.errorbar(np.log(N), S_A, yerr=sigma_total, fmt='o', capsize=3, label='DMRG data')
ax.set_xlabel('ln(N)')
ax.set_ylabel('S_A')
ax.set_title('Entanglement Entropy vs Chain Length')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Error budget
ax = axes[1]
ax.plot(N, sigma_num, 'o-', label='Numerical uncertainty')
ax.plot(N, sigma_trunc, 's-', label='Truncation error')
ax.plot(N, sigma_total, '^-', label='Total error')
ax.set_xlabel('N')
ax.set_ylabel('Error')
ax.set_title('Error Budget')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Bayesian Model Comparison

In [ ]:
# Initialize analyzer
analyzer = BayesianScalingAnalyzer(N, S_A, sigma_num, sigma_trunc)

# Compare all models
results = analyzer.compare_models()
analyzer.print_comparison_table(results)

## 4. Best Model Fit (1/N² - BCFT Prediction)

In [ ]:
# Extract best model results
best = results['1/N^2 (BCFT)']
c_fit, g_fit, B_fit = best['params']
c_err, g_err, B_err = best['errors']

print("Best Fit Parameters (1/N² model):")
print("=" * 50)
print(f"c (central charge) = {c_fit:.4f} ± {c_err:.4f}")
print(f"g (boundary entropy) = {g_fit:.4f} ± {g_err:.4f}")
print(f"B (correction amplitude) = {B_fit:.4f} ± {B_err:.4f}")
print(f"\nχ²/dof = {best['chi2_reduced']:.3f}")
print(f"BIC = {best['bic']:.1f}")

## 5. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap resampling
boot_params = analyzer.bootstrap_fit(
    analyzer.model_1_over_N2, 
    n_bootstrap=1000, 
    seed=42
)

ci = analyzer.compute_confidence_intervals(boot_params)

print("Bootstrap 95% Confidence Intervals:")
print("=" * 50)
param_names = ['c', 'g', 'B']
for i, name in enumerate(param_names):
    p = best['params'][i]
    ci_l = ci[f'param_{i}']['ci_lower']
    ci_u = ci[f'param_{i}']['ci_upper']
    print(f"{name} = {p:.4f} (95% CI: [{ci_l:.4f}, {ci_u:.4f}])")

## 6. Visualization of Fit

In [ ]:
# Generate fit curve
N_fit = np.linspace(N.min(), N.max(), 200)
S_A_fit = analyzer.model_1_over_N2(N_fit, c_fit, g_fit, B_fit)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Data + Fit
ax = axes[0]
ax.errorbar(N, S_A, yerr=sigma_total, fmt='o', capsize=3, label='DMRG data')
ax.plot(N_fit, S_A_fit, 'r-', label=f'1/N² fit (c={c_fit:.3f})')
ax.set_xlabel('N')
ax.set_ylabel('S_A')
ax.set_title('Entanglement Entropy: Data vs BCFT Fit')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Residuals
ax = axes[1]
residuals = S_A - analyzer.model_1_over_N2(N, c_fit, g_fit, B_fit)
ax.errorbar(N, residuals, yerr=sigma_total, fmt='o', capsize=3)
ax.axhline(y=0, color='r', linestyle='--')
ax.set_xlabel('N')
ax.set_ylabel('Residuals')
ax.set_title('Fit Residuals')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary

**Key Results:**
- Central charge: c = 1.008 (95% CI: [0.976, 1.040])
- Boundary entropy: g = 0.352 (95% CI: [0.330, 0.374])
- Evidence ratio: K ≈ 1800:1 favoring 1/N² over 1/N

**Conclusion:** The BCFT-predicted 1/N² correction is decisively confirmed.